In [0]:
sales = spark.read.table("de_mini_project.silver.sales")
customer = spark.read.table("de_mini_project.silver.customer")
returns = spark.read.table("de_mini_project.silver.return")
stores = spark.read.table("de_mini_project.silver.stores")
display(sales)
display(customer)
display(returns)
display(stores)

In [0]:
#created an iterable to pass only desired data into dataframe
sales_iterable = sales.select("transaction_id","sku_id","customer_id","store_id", "qty_sold", "unit_price","date").collect()

In [0]:
from pyspark.sql.functions import lit

fact_sales = spark.createDataFrame(sales_iterable, ["transaction_id","sku_id","customer_id","store_id", "qty_sold", "unit_price","date"])

In [0]:
returns = returns.withColumnRenamed("date","return_date")
#Left join with returns table to get which purchases were returned
fact_sales = fact_sales.join(returns,["transaction_id"], "left")
fact_sales = fact_sales.drop("reason")
display(fact_sales)

In [0]:
fact_sales.write.mode("overwrite").saveAsTable("de_mini_project.gold.fact_sales")

In [0]:
customer.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.gold.dim_customer")
dim_returns = returns.drop("return_date","transaction_id")
dim_returns.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.gold.dim_returns")



In [0]:
stores.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.gold.dim_stores")